Adding score

In [37]:
from langfuse import get_client

langfuse = get_client()
logged_scores = 0

with langfuse.start_as_current_observation(
    as_type="span",
    name="DeepEval-HR-Policy-Evaluation",
    input={"source": "DeepEval evaluation_results"}
) as evaluation_span:

    for test_result in evaluation_results.test_results:
        print(f"\nTest case: {test_result.name}")

        for metric in test_result.metrics_data:
            score = metric.score

            print(f"{metric.name}: {score}")

            if score is None:
                print("Skipped: score is None")
                continue

            evaluation_span.score_trace(
                name=metric.name,
                value=float(score),
                data_type="NUMERIC",
                comment=f"DeepEval evaluation for {test_result.name}"
            )

            logged_scores += 1

langfuse.flush()

print(f"\nScores submitted: {logged_scores}")


Test case: test_case_0
Answer Relevancy: 1.0
Faithfulness: 1.0
Correctness [GEval]: 0.7851952801968312
Completeness [GEval]: 0.7320821300824607
Clarity [GEval]: 0.7060086650174008
Consistency [GEval]: 0.7095349464899109

Scores submitted: 6


In [38]:
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig
from deepeval.test_case import LLMTestCase

def evaluate_rag_result(result, expected_answer):
    test_case = LLMTestCase(
        input=result["question"],
        actual_output=result["answer"],
        expected_output=expected_answer,
        retrieval_context=[
            item["text"] for item in result["contexts"]
        ],
    )

    metrics = [
        answer_relevancy,
        faithfulness,
        correctness,
        completeness,
        clarity,
        consistency,
    ]

    evaluation_results = evaluate(
        test_cases=[test_case],
        metrics=metrics,
        async_config=AsyncConfig(run_async=False),
    )

    print("\n===== RAG EVALUATION COMPLETED =====")
    print(evaluation_results)

    return evaluation_results


# Generate a monitored RAG response first
result = monitor_rag(
    question="What are the regular office hours?",
    retrieve_fn=retrieve_context,
    generate_fn=generate_answer,
)

print("Generated answer:", result["answer"])

# Evaluate the generated answer against the HR policy
expected_answer = (
    "Regular office hours are 9 AM to 6 PM Monday to Friday, "
    "excluding holidays, and 9 AM to 2 PM on Saturday. "
    "Employees receive at least a 30-minute lunch break, "
    "generally between 1 PM and 1:30 PM."
)

evaluation_results = evaluate_rag_result(
    result,
    expected_answer,
)

print("\nEvaluation object returned successfully.")


NameError: name 'monitor_rag' is not defined

In [36]:
print(dir(evaluation_results))

['__abstractmethods__', '__annotate_func__', '__annotations__', '__annotations_cache__', '__class__', '__class_getitem__', '__class_vars__', '__copy__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__fields__', '__fields_set__', '__firstlineno__', '__format__', '__ge__', '__get_pydantic_core_schema__', '__get_pydantic_json_schema__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__pretty__', '__private_attributes__', '__pydantic_complete__', '__pydantic_computed_fields__', '__pydantic_core_schema__', '__pydantic_custom_init__', '__pydantic_decorators__', '__pydantic_extra__', '__pydantic_extra_info__', '__pydantic_fields__', '__pydantic_fields_set__', '__pydantic_generic_metadata__', '__pydantic_init_subclass__', '__pydantic_on_complete__', '__pydantic_parent_namespace__', '__pydantic_post_init__', '__pydantic_private__', '__pyda

In [33]:
print(type(evaluation_results))
print(evaluation_results)

<class 'deepeval.evaluate.types.EvaluationResult'>
test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.7, success=True, score=1.0, reason='The score is 1.00 because the response directly addresses the question about regular office hours without including any irrelevant information.', strict_mode=False, flaky=False, evaluation_model='gpt-4.1-mini', error=None, evaluation_cost=0.0006120000000000001, input_tokens=1094, output_tokens=109, verbose_logs='Statements:\n[\n    "Office hours are 9 AM to 6 PM Monday to Friday.",\n    "Office hours are 9 AM to 2 PM on Saturday."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    },\n    {\n        "verdict": "borderline",\n        "reason": "The input asks for regular office hours, which typically refer to weekdays; Saturday hours may be considered additional or exceptional."\n    }\n]'), MetricData(name='Faithfulness', threshold=0.7, success=True

In [32]:
langfuse = get_client()

In [31]:
print(dir(evaluation_results))

['__abstractmethods__', '__annotate_func__', '__annotations__', '__annotations_cache__', '__class__', '__class_getitem__', '__class_vars__', '__copy__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__fields__', '__fields_set__', '__firstlineno__', '__format__', '__ge__', '__get_pydantic_core_schema__', '__get_pydantic_json_schema__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__pretty__', '__private_attributes__', '__pydantic_complete__', '__pydantic_computed_fields__', '__pydantic_core_schema__', '__pydantic_custom_init__', '__pydantic_decorators__', '__pydantic_extra__', '__pydantic_extra_info__', '__pydantic_fields__', '__pydantic_fields_set__', '__pydantic_generic_metadata__', '__pydantic_init_subclass__', '__pydantic_on_complete__', '__pydantic_parent_namespace__', '__pydantic_post_init__', '__pydantic_private__', '__pyda

In [30]:
print(type(evaluation_results))
print(evaluation_results)

<class 'deepeval.evaluate.types.EvaluationResult'>
test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.7, success=True, score=1.0, reason='The score is 1.00 because the response directly addresses the question about regular office hours without including any irrelevant information.', strict_mode=False, flaky=False, evaluation_model='gpt-4.1-mini', error=None, evaluation_cost=0.0006120000000000001, input_tokens=1094, output_tokens=109, verbose_logs='Statements:\n[\n    "Office hours are 9 AM to 6 PM Monday to Friday.",\n    "Office hours are 9 AM to 2 PM on Saturday."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    },\n    {\n        "verdict": "borderline",\n        "reason": "The input asks for regular office hours, which typically refer to weekdays; Saturday hours may be considered additional or exceptional."\n    }\n]'), MetricData(name='Faithfulness', threshold=0.7, success=True

In [29]:
def log_deepeval_score(name, score, trace_id=None):
    """
    Log a DeepEval score to Langfuse.

    If trace_id is supplied, associate the score with that trace.
    """
    kwargs = {
        "name": name,
        "value": float(score),
        "data_type": "NUMERIC",
    }

    if trace_id:
        kwargs["trace_context"] = {"trace_id": trace_id}

    langfuse.create_score(**kwargs)

    print(f"Logged {name}: {score}")

In [77]:
answer = monitored_rag("What are the Emergency Leave?")
print(answer)

2026-10-15 09:45:10,903 | INFO | HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-10-15 09:45:12,054 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Emergency Leave (EL) entitles all employees to 12 days of leave that can be availed either for the employee’s marriage or in the event of the death of his/her family members, specifically father, mother, spouse, and children.


In [55]:
answer = monitored_rag("What are the Maternity Leave?")
print(answer)

2026-10-14 11:36:42,517 | INFO | HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-10-14 11:36:43,774 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


According to the Maternity Benefit Act, female employees are entitled to a maximum of 12 weeks (90 days) of maternity leave.


In [28]:
answer = monitored_rag("What are the regular office hours?")
print(answer)

The regular office hours are 9 a.m. to 6 p.m. from Monday to Friday (excluding holidays), and on Saturday the office timing is 9 a.m. to 2 p.m.


    Setting up monitoring using Languse

In [27]:
from langfuse import get_client
import time

langfuse = get_client()

def monitored_rag(question):

    with langfuse.start_as_current_observation(
        as_type="span",
        name="HR-Policy-RAG",
        input={"question": question}
    ) as trace:

        start = time.perf_counter()

        # Step 1: Retrieve documents
        with langfuse.start_as_current_observation(
            as_type="span",
            name="ChromaDB-Retrieval"
        ) as retrieval:

            docs = vectorstore.similarity_search(question, k=4)

            context = "\n\n".join(
                doc.page_content for doc in docs
            )

            retrieval.update(
                output={"retrieved_chunks": len(docs)}
            )

        # Step 2: Generate answer
        prompt = f"""
Answer the question using only the provided context.

Context:
{context}

Question:
{question}
"""

        with langfuse.start_as_current_observation(
            as_type="generation",
            name="Open AI",
            input={"prompt": prompt}
        ) as generation:

            response = llm.invoke(prompt)

            answer = response.content

            generation.update(output=answer)

        trace.update(
            output={"answer": answer},
            metadata={
                "total_latency_ms": round(
                    (time.perf_counter() - start) * 1000, 2
                )
            }
        )

    langfuse.flush()
    return answer

Checking LLM EXIST and vectorstore EXIST

In [26]:
print("LLM exists:", "llm" in globals())
print("Vectorstore exists:", "vectorstore" in globals())

LLM exists: True
Vectorstore exists: True


Setting up the connection

In [52]:
from dotenv import load_dotenv
from langfuse import get_client

load_dotenv(override=True)

langfuse = get_client()

with langfuse.start_as_current_observation(
    as_type="span",
    name="langfuse-connection-test",
    input={"test": "HR policy RAG monitoring"}
) as span:
    span.update(
        output={"status": "Connection test completed"}
    )

langfuse.flush()

print("Langfuse test trace sent.")

Langfuse test trace sent.


Test cases

In [51]:
from deepeval.test_case import LLMTestCase

# Normal test case
test_case = LLMTestCase(
    input="What is the company's leave policy?",
    actual_output="Employees receive 20 days of annual leave every year.",
    retrieval_context=[
        "Employees receive 20 days of annual leave every year."
    ]
)

# Deliberate hallucination test case
bad_test_case = LLMTestCase(
    input="What is the company's leave policy?",
    actual_output="Employees receive 100 days of annual leave every year.",
    retrieval_context=[
        "Employees receive 20 days of annual leave every year."
    ]
)

# Combine both test cases
test_cases = [test_case, bad_test_case]

print(f"Created {len(test_cases)} test cases successfully!")

Created 2 test cases successfully!


In [50]:
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig


In [49]:
from deepeval import evaluate
from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    GEval,
)
from deepeval.test_case import LLMTestCaseParams
from deepeval.evaluate import AsyncConfig

# Configuration
MODEL = "gpt-4.1-mini"
THRESHOLD = 0.7

# Select only the metrics you need
SELECTED_METRICS = [
    #"faithfulness"
     "answer_relevancy",
    # "correctness",
    # "completeness",
    # "clarity",
    # "consistency",
]

# Create only the selected metrics
metrics = []

if "faithfulness" in SELECTED_METRICS:
    metrics.append(
        FaithfulnessMetric(
            threshold=THRESHOLD,
            model=MODEL
        )
    )

if "answer_relevancy" in SELECTED_METRICS:
    metrics.append(
        AnswerRelevancyMetric(
            threshold=THRESHOLD,
            model=MODEL
        )
    )

if "correctness" in SELECTED_METRICS:
    metrics.append(
        GEval(
            name="Correctness",
            criteria=(
                "Evaluate whether the actual output is "
                "factually correct compared with the "
                "expected output."
            ),
            evaluation_params=[
                LLMTestCaseParams.ACTUAL_OUTPUT,
                LLMTestCaseParams.EXPECTED_OUTPUT,
            ],
            threshold=THRESHOLD,
            model=MODEL,
        )
    )

if "completeness" in SELECTED_METRICS:
    metrics.append(
        GEval(
            name="Completeness",
            criteria=(
                "Evaluate whether the answer covers all "
                "important information required by the question."
            ),
            evaluation_params=[
                LLMTestCaseParams.INPUT,
                LLMTestCaseParams.ACTUAL_OUTPUT,
                LLMTestCaseParams.EXPECTED_OUTPUT,
            ],
            threshold=THRESHOLD,
            model=MODEL,
        )
    )

if "clarity" in SELECTED_METRICS:
    metrics.append(
        GEval(
            name="Clarity",
            criteria=(
                "Evaluate whether the answer is clear, "
                "understandable, and well organized."
            ),
            evaluation_params=[
                LLMTestCaseParams.INPUT,
                LLMTestCaseParams.ACTUAL_OUTPUT,
            ],
            threshold=THRESHOLD,
            model=MODEL,
        )
    )

if "consistency" in SELECTED_METRICS:
    metrics.append(
        GEval(
            name="Consistency",
            criteria=(
                "Evaluate whether the answer is internally "
                "consistent and contains no contradictions."
            ),
            evaluation_params=[
                LLMTestCaseParams.ACTUAL_OUTPUT,
            ],
            threshold=THRESHOLD,
            model=MODEL,
        )
    )

if not metrics:
    raise ValueError("Select at least one metric.")

print("Selected metrics:", SELECTED_METRICS)

evaluation_results = evaluate(
    test_cases=test_cases,
    metrics=metrics,
    async_config=AsyncConfig(run_async=False),
)

print("DeepEval evaluation completed!")
print(evaluation_results)


Selected metrics: ['answer_relevancy']


C:\Users\Shubham\AppData\Local\Temp\ipykernel_9584\2892019462.py:7: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1-mini, strict=False, 
async_mode=False)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                     ┃ Average Score         ┃ Pass Rate                                     ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Answer Relevancy           │ 1.00                  │ 100.00% | passed=1 | failed=0                 │ 1         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=13783769;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 3.45s | token cost: 0.0006328000000000001 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

DeepEval evaluation completed!
test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.7, success=True, score=1.0, reason='The score is 1.00 because the response directly addresses the question about regular office hours without including any irrelevant information. It is concise and fully relevant.', strict_mode=False, flaky=False, evaluation_model='gpt-4.1-mini', error=None, evaluation_cost=0.0006328000000000001, input_tokens=1094, output_tokens=122, verbose_logs='Statements:\n[\n    "Office hours are 9 AM to 6 PM Monday to Friday.",\n    "Office hours are 9 AM to 2 PM on Saturday."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    },\n    {\n        "verdict": "borderline",\n        "reason": "The statement provides additional information about office hours on Saturday, which may be relevant but is not part of the regular Monday to Friday office hours requested."\n    }\n]')], conversa

In [ ]:
# from deepeval import evaluate
# from deepeval.evaluate import AsyncConfig

# evaluation_results = evaluate(
#     test_cases=test_cases,
#     metrics=[
#         answer_relevancy,
#         faithfulness,
#         correctness,
#         completeness,
#         clarity,
#         consistency,
#     ],
#     async_config=AsyncConfig(run_async=False),
# )

# print("DeepEval evaluation completed!")
# print(evaluation_results)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1-mini, strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4.1-mini, strict=False, async_mode=False)...

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-4.1-mini, strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Completeness [GEval] Metric! (using gpt-4.1-mini, strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Clarity [GEval] Metric! (using gpt-4.1-mini, strict=False, async_mode=False)...

...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 6 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                         ┃ Average Score        ┃ Pass Rate                                   ┃ Total    │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━ │
│  Answer Relevancy               │ 1.00                 │ 100.00% | passed=1 | failed=0               │ 1        │
│  Faithfulness                   │ 1.00                 │ 100.00% | passed=1 | failed=0               │ 1        │
│  Correctness [GEval]            │ 0.79                 │ 100.00% | passed=1 | failed=0               │ 1        │
│  Completeness [GEval]           │ 0.73                 │ 100.00% | passed=1 | failed=0               │ 1        │
│  Clarity [GEval]                │ 0.71                 │ 100.00% | passed=1 | failed=0               │ 1        │
│  Consistency [GEval]            │ 0.71                 │ 100.00% | passed=1 | failed=0               │ 1        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=13783763;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 27.96s | token cost: 0.003312400000000001 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

DeepEval evaluation completed!
test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.7, success=True, score=1.0, reason='The score is 1.00 because the response directly addresses the question about regular office hours without including any irrelevant information.', strict_mode=False, flaky=False, evaluation_model='gpt-4.1-mini', error=None, evaluation_cost=0.0006120000000000001, input_tokens=1094, output_tokens=109, verbose_logs='Statements:\n[\n    "Office hours are 9 AM to 6 PM Monday to Friday.",\n    "Office hours are 9 AM to 2 PM on Saturday."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    },\n    {\n        "verdict": "borderline",\n        "reason": "The input asks for regular office hours, which typically refer to weekdays; Saturday hours may be considered additional or exceptional."\n    }\n]'), MetricData(name='Faithfulness', threshold=0.7, success=True, score=1.0, reason=

Test cases.

In [48]:
from deepeval.test_case import LLMTestCase

test_cases = [
    LLMTestCase(
        input="What are the regular office hours?",
        actual_output=(
            "Office hours are 9 AM to 6 PM Monday to Friday "
            "and 9 AM to 2 PM on Saturday."
        ),
        expected_output=(
            "The company HR policy specifies office hours of "
            "9 AM to 6 PM Monday to Friday, excluding holidays, "
            "and 9 AM to 2 PM on Saturday."
        ),
        retrieval_context=[
            "Regular office hours are 9 a.m. to 6 p.m. Monday "
            "to Friday, excluding holidays. Saturday office "
            "hours are 9 a.m. to 2 p.m."
        ],
    )
]

In [ ]:
# from deepeval import evaluate
# from deepeval.evaluate import AsyncConfig
# from deepeval.test_case import LLMTestCase

# sample_test_case = LLMTestCase(
#     input="What are the regular office hours according to the company HR policy?",
#     actual_output=(
#         "The regular office hours are 9 AM to 6 PM from Monday "
#         "to Friday, excluding holidays. On Saturday, office hours "
#         "are 9 AM to 2 PM. Employees receive at least a 30-minute "
#         "lunch break, generally between 1 PM and 1:30 PM."
#     ),
#     expected_output=(
#         "Regular office hours are 9 AM to 6 PM Monday to Friday, "
#         "excluding holidays, and 9 AM to 2 PM on Saturday. "
#         "Employees receive at least a 30-minute lunch break, "
#         "generally between 1 PM and 1:30 PM."
#     ),
# )

# evaluation_results = evaluate(
#     test_cases=[sample_test_case],
#     metrics=[
#         correctness,
#         completeness,
#         clarity,
#         consistency
#     ],
# #     async_config=AsyncConfig(run_async=False),
# # )

# print("Correctness and Completeness evaluation finished!")
# print(evaluation_results)

SyntaxError: '(' was never closed (404530386.py, line 21)

Custom Matrix

In [47]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

MODEL = "gpt-4.1-mini"
THRESHOLD = 0.7

criteria = {
    "correctness": (
        "Evaluate whether the actual output is factually accurate "
        "and answers the input correctly. Compare the actual output "
        "with the expected output. Penalize factual errors, incorrect "
        "claims, and contradictions. Do not penalize differences in "
        "wording when the meaning is correct."
    ),
    "completeness": (
        "Evaluate whether the actual output includes all important "
        "information required by the input and covered by the expected "
        "output. Penalize missing key facts, steps, explanations, or "
        "requirements. Give a high score when all important information "
        "is covered, even if the wording differs."
    ),
    "clarity": (
        "Evaluate whether the actual output is clear, concise, "
        "well-organized, and easy to understand."
    ),
    "consistency": (
        "Evaluate whether the actual output is logically consistent "
        "with the expected output and contains no contradictions."
    ),
}

custom_metrics = {
    name: GEval(
        name=name.title(),
        criteria=criterion,
        evaluation_params=[
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT,
            LLMTestCaseParams.EXPECTED_OUTPUT,
        ],
        threshold=THRESHOLD,
        model=MODEL,
        strict_mode=False,
    )
    for name, criterion in criteria.items()
}

correctness = custom_metrics["correctness"]
completeness = custom_metrics["completeness"]
clarity = custom_metrics["clarity"]
consistency = custom_metrics["consistency"]

print("Custom metrics ready:", list(custom_metrics.keys()))

Custom metrics ready: ['correctness', 'completeness', 'clarity', 'consistency']


C:\Users\Shubham\AppData\Local\Temp\ipykernel_9584\1555735307.py:2: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


Create Faithfulness metric & Answer Relevancy metric

In [19]:
from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric
)

MODEL = "gpt-4.1-mini"
THRESHOLD = 0.7

faithfulness = FaithfulnessMetric(
    threshold=THRESHOLD,
    model=MODEL
)

answer_relevancy = AnswerRelevancyMetric(
    threshold=THRESHOLD,
    model=MODEL
)

print("Faithfulness ready")
print("Answer Relevancy ready")

Faithfulness ready
Answer Relevancy ready


DeepEval test cases creation

In [46]:
test_cases = []

for result in rag_results:

    test_case = LLMTestCase(
        input=result["question"],
        actual_output=result["answer"],
        retrieval_context=result["context"]
    )

    test_cases.append(test_case)

print(
    "DeepEval test cases:",
    len(test_cases)
)


DeepEval test cases: 4


Running questions through LangGraph

In [45]:
rag_results = []

for question in evaluation_questions:

    result = rag_graph.invoke({
        "question": question,
        "context": [],
        "answer": ""
    })

    rag_results.append(result)

    print("=" * 80)

    print("QUESTION:")
    print(question)

    print("\nANSWER:")
    print(result["answer"])


QUESTION:
What is the company's leave policy?

ANSWER:
The company's leave policy is as follows:

- Leave is permitted at the discretion of management and cannot be claimed as a right.
- Employees must obtain prior approval from their HOD at least 3 days in advance using the prescribed form to avail any type of leave, unless sanctioned by the CEO.
- If an employee remains absent beyond the sanctioned or extended leave period, two days' salary will be deducted for every one day absent.
- Probationers are not eligible for casual leave during the probation period.

Specific leave types:
- Privilege Leave (PL): 15 days including Sundays and holidays, applicable only for Management and above after confirmation or completion of one year of service. Applications must be made at least 1 month prior to leave. More than one management member cannot avail privilege leave simultaneously. Extension beyond sanctioned leave requires CEO approval. Unavailed privilege leave will not be carried forward 

In [53]:

from pathlib import Path

# Read questions from the text file
file_path = Path("evaluation_questions.txt")

evaluation_questions = [
    line.strip()
    for line in file_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

print(f"Total questions loaded: {len(evaluation_questions)}")

# Evaluate each question
for i, question in enumerate(evaluation_questions, start=1):

    print(f"\n{'=' * 60}")
    print(f"Question {i}: {question}")
    print("=" * 60)

    # Retrieve relevant documents
    retrieved_docs = retriever.invoke(question)

    # Convert retrieved documents into text
    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    # Create the prompt
    prompt = f"""
Answer the question using only the context below.
If the answer is not present, say "I don't know."

Question:
{question}

Context:
{context}

Answer:
"""

    # 3. Generate the answer
    response = llm.invoke(prompt)

    # 4. Display the answer
    print("Answer:")
    print(response.content if hasattr(response, "content") else response)

Total questions loaded: 4

Question 1: What is the company's leave policy?
Answer:
The company's leave policy is as follows:

- Leave is granted at the discretion of management and cannot be claimed as a right. Leave can be refused or revoked if necessary due to work exigencies.
- Employees must obtain prior approval from their HOD at least 3 days in advance before taking any type of leave, unless sanctioned by the CEO.
- If an employee remains absent beyond the approved leave period, two days' salary will be deducted for every one day absent.
- Probationers are not eligible for casual leave during their probation.
- Privilege Leave (PL) of 15 days (including Sundays and holidays) is available for management and above after completing one year of service and confirmation. PL must be applied for at least 1 month in advance.
- More than one management member cannot take privilege leave simultaneously.
- Extension of privilege leave beyond the sanctioned period requires CEO approval.
- Un

several questions added in file

In [42]:
from pathlib import Path

# Read questions only from the text file
file_path = Path("evaluation_questions.txt")

evaluation_questions = [
    line.strip()
    for line in file_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

# Display the questions
for i, question in enumerate(evaluation_questions, start=1):
    print(f"{i}. {question}")


1. What is the company's leave policy?
2. How many annual leave days are provided?
3. What is the work-from-home policy?
4. What is the resignation policy?


In [ ]:
# evaluation_questions = [
#     "What is the company's leave policy?",
#     "How many annual leave days are provided?",
#     "What is the work-from-home policy?",
#     "What is the resignation policy?"
# ]

Display the retrieved context

In [15]:
print("========== RETRIEVED CONTEXT ==========")

for i, context in enumerate(result["context"]):

    print(f"\n--- Chunk {i + 1} ---")

    print(context)


========== RETRIEVED CONTEXT ==========

--- Chunk 1 ---
PROVIDENT FUND & ESIC
Every eligible employee of the company is entitled to receive the benefit of Provident Fund,
ESIC & Insurance.
TDS and Professional Tax will be deducted as per government law.
Employees must update their details of nomination for settling the PF, ESIC & Insurance
claims.
LEAVE RULES
GENERAL LEAVE RULES
Employees will be permitted to take leave at the discretion of the management. Leave
cannot be claimed as a matter of right. The sanctioning authority reserves the
discretion to refuse or revoke leave at any time, if necessary due to exigencies of
company’s work.
1) Each employee must take prior approval/proper authorization (through prescribed
form) from their HOD’s in advance before 3 days to avail any type of leave.

--- Chunk 2 ---
unless sanctioned by CEO.
5. Unavailed privilege leave will not be carried forward or accumulated.
MATERNITY LEAVE (ML)
According to the Maternity Benefit Act female employees a

Test the RAG application

In [14]:
question = "What is the company's leave policy?"

result = rag_graph.invoke({
    "question": question,
    "context": [],
    "answer": ""
})

print("QUESTION:")
print(result["question"])

print("\nANSWER:")
print(result["answer"])


QUESTION:
What is the company's leave policy?

ANSWER:
The company's leave policy is as follows:

- Leave is permitted at the discretion of management and cannot be claimed as a right. The sanctioning authority can refuse or revoke leave if necessary due to company work exigencies.
- Employees must obtain prior approval from their HOD at least 3 days in advance using the prescribed form before availing any type of leave, unless sanctioned by the CEO.
- If an employee remains absent beyond the sanctioned leave or its extension, two days' salary will be deducted for every one day of absence.
- Probationers are not eligible for casual leave during the probation period.
- Privilege Leave (PL) rules for Management and above:
  1. 15 days of Privilege leave (including Sundays and Holidays) is applicable.
  2. The employee must apply at least 1 month prior to the leave.
  3. Only upon confirmation or completion of one year of service is an employee eligible.
  4. More than one management memb

Build the LangGraph

In [13]:
builder = StateGraph(RAGState)

builder.add_node(
    "retrieve",
    retrieve_node
)

builder.add_node(
    "generate",
    generate_node
)

builder.add_edge(
    START,
    "retrieve"
)

builder.add_edge(
    "retrieve",
    "generate"
)

builder.add_edge(
    "generate",
    END
)

rag_graph = builder.compile()

print("LangGraph created successfully")


LangGraph created successfully


Create the generation node

In [12]:
def generate_node(state: RAGState):

    question = state["question"]

    context = "\n\n".join(
        state["context"]
    )

    prompt = f"""
You are a helpful RAG assistant.

Answer the question using ONLY
the context provided below.

Do not make up information.

If the answer is not available
in the context, say:

"I don't know based on the provided documents."

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

    response = llm.invoke(prompt)

    return {
        "answer": response.content
    }


Create the retrieval node

In [11]:
def retrieve_node(state: RAGState):

    question = state["question"]

    docs = retriever.invoke(question)

    context = [
        doc.page_content
        for doc in docs
    ]

    return {
        "context": context
    }

Now create the LangGraph state

In [10]:
class RAGState(TypedDict):
    question: str
    context: List[str]
    answer: str

Create the LLM

In [9]:
llm = ChatOpenAI(
    model="gpt-4.1-mini",
   # temperature=0
)

print("LLM ready")

LLM ready


Create embeddings

In [8]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 4
    }
)

print("Vector database created")

Vector database created


In [7]:
print(chunks[0].page_content)


1
Kamaiah Engineering Services Private Limited™
ISO 9001:2008 Certified
HR POLICY
www.kamaiahgroup.com


Split the PDF

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(
    documents
)

print("Number of chunks:", len(chunks))

Number of chunks: 55


Loading PDF


In [5]:
if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"PDF not found: {PDF_PATH}"
    )

loader = PyPDFLoader(
    str(PDF_PATH)
)

documents = loader.load()

print("Number of pages:", len(documents))

Number of pages: 20


Load your API key

In [4]:
load_dotenv(ENV_PATH)

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY is missing from .env"
    )

print("OpenAI API key loaded")

OpenAI API key loaded


Project folder 

In [ ]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"

PDF_PATH = DATA_DIR / "company_policy.pdf"

ENV_PATH = PROJECT_ROOT / ".env"

print("Project root:", PROJECT_ROOT)
print("PDF path:", PDF_PATH)


Project root: c:\Users\Shubham\DeepEval_RAG_Demo
PDF path: c:\Users\Shubham\DeepEval_RAG_Demo\data\company_policy.pdf


In [ ]:
from langchain_community.vectorstores import FAISS
from langgraph.graph import StateGraph, START, END

from deepeval import evaluate
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    GEval
)

print("Libraries loaded successfully")

Libraries loaded successfully


C:\Users\Shubham\AppData\Local\Temp\ipykernel_9584\3426634594.py:5: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


Import OS


In [ ]:
import os
from pathlib import Path
from typing import TypedDict, List

from dotenv import load_dotenv

from deepeval.metrics import GEval

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langgraph.graph import StateGraph, START, END

from deepeval import evaluate
from deepeval.test_case import LLMTestCase

from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric
)

print("Libraries loaded successfully")


C:\Users\Shubham\AppData\Local\Temp\ipykernel_9584\274356874.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Libraries loaded successfully
